In [0]:
from pyspark.sql.functions import lit
import json

In [0]:
def retrieve_last_n_posts_from_followee(spark, db, table_names, run_date, last_n=10, to_pandas=False):
    sdf = spark.sql("""
                    select user_id,
                    collect_set(post_id) as post_id
                    from(
                        select user_id,
                        followee_id,
                        post_id,
                        ROW_NUMBER() OVER (
                        PARTITION BY user_id, followee_id
                        ORDER BY partition_date DESC
                        ) AS rn
                        from {db}.{follower_followee_post}
                        where partition_date <= '{run_date}'
                    )
                    where rn <= {last_n}
                    group by 1
                    """.format(db = db,
                               follower_followee_post = table_names["follower_followee_post"],
                               run_date = run_date,
                               last_n = last_n)
    )
    return sdf.toPandas() if to_pandas else sdf

In [0]:
def select_active_users(spark, run_date, lookback_win, db, table_names, to_pandas=False):
    sdf = spark.sql("""
                    select distinct user_id 
                    from {db}.{user_attributes}
                    where community_visits > 0 
                    and partition_date between date_sub('{run_date}', {lookback_win}) and '{run_date}'
                    """.format(db = db,
                               user_attributes = table_names["user_attributes"],
                               run_date = run_date,
                               lookback_win = lookback_win)
    )
    return sdf.toPandas() if to_pandas else sdf

def filter_active_users(df, user_set_df):
    # keeps rows in df where user_id exists in user_set_df
    filtered_df = df.join(user_set_df.select("user_id"), on="user_id", how="left_semi")
    return filtered_df 

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)
last_n = model_config['last_n_posts']
lookback_win = model_config['active_user_window']

In [0]:
sdf = retrieve_last_n_posts_from_followee(spark, db, table_names, run_date, last_n)

In [0]:
user_set_sdf = select_active_users(spark, run_date, lookback_win, db, table_names)
sdf = filter_active_users(sdf, user_set_sdf)

In [0]:
sdf = sdf.withColumn("partition_date", lit(run_date))

In [0]:
# Write data to table
sdf.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(db+'.'+table_names['follower_retrieval'])